# Handwritten English Alphabet Recognition using CNN

**Phase 2 - Code Implementation** | Dataset: A-Z Handwritten Alphabets (CSV) | [Kaggle dataset](https://www.kaggle.com/datasets/sachinpatel21/az-handwritten-alphabets-in-csv-format)

## 0. Setup and reproducibility

In [ ]:
import os, random, glob, json, gc
os.environ["PYTHONHASHSEED"] = "0"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                             recall_score, roc_curve, auc)
from sklearn.preprocessing import label_binarize
from sklearn.utils.class_weight import compute_class_weight

# ---- reproducibility ----
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

# ---- configuration (mirrors the proposal's hyperparameters) ----
BATCH         = 128      # batch size
EPOCHS        = 20       # max epochs for the custom CNN (early stopping may stop sooner)
IMG_TL        = 32       # input size for MobileNetV2 (its minimum is 32x32)
BATCH_TL      = 128
HEAD_EPOCHS   = 5        # transfer learning: train the new head
FT_EPOCHS     = 3        # transfer learning: fine-tune top layers
SAMPLE_FRACTION   = 1.0  # set < 1.0 (e.g. 0.2) for a quick test run on part of the data
USE_CLASS_WEIGHTS = True # handle imbalance by weighting rare letters more
DO_FINETUNE       = True

AUTOTUNE = tf.data.AUTOTUNE
LETTERS  = [chr(65 + i) for i in range(26)]   # 0->A ... 25->Z
NUM_CLASSES = 26

# ---- output folders (Kaggle writes to /kaggle/working) ----
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
FIG_DIR   = os.path.join(WORK, "figures"); os.makedirs(FIG_DIR, exist_ok=True)    # figures saved as PDF
TABLE_DIR = os.path.join(WORK, "tables");  os.makedirs(TABLE_DIR, exist_ok=True)  # tables saved as CSV
MODEL_DIR = os.path.join(WORK, "models");  os.makedirs(MODEL_DIR, exist_ok=True)

print("TensorFlow:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices("GPU"))

## 1. Dataset loading

The dataset is a single CSV with no header. Each row is one image: column 0 is the label
(an integer 0-25, where 0 = A and 25 = Z) and the remaining 784 columns are the flattened
28x28 grayscale pixels (0-255). We read it with `dtype=uint8` to keep memory low.

In [ ]:
# Find the CSV inside the Kaggle input folder (robust to exact folder name)
csv_candidates = glob.glob("/kaggle/input/**/*.csv", recursive=True)
print("CSV files found:", csv_candidates)

csv_path = None
for c in csv_candidates:
    if "handwritten" in c.lower() or "a_z" in c.lower().replace(" ", "_"):
        csv_path = c; break
if csv_path is None and csv_candidates:
    csv_path = csv_candidates[0]
assert csv_path is not None, "Could not find the dataset CSV under /kaggle/input"
print("Using:", csv_path)

# Read as uint8 (labels 0-25 and pixels 0-255 all fit in a byte) -> ~290 MB
df = pd.read_csv(csv_path, header=None, dtype=np.uint8)
print("Raw shape:", df.shape)   # expect (~372451, 785)

# Split into labels and pixel features
labels = df.iloc[:, 0].to_numpy()
X = df.iloc[:, 1:].to_numpy()
del df; gc.collect()

print("Labels range:", labels.min(), "to", labels.max(), "| unique:", len(np.unique(labels)))
print("Pixels shape:", X.shape, "| min/max:", X.min(), X.max())

## 2. Data exploration (EDA)

In [ ]:
# ---- class distribution ----
counts = pd.Series(labels).value_counts().sort_index()
counts.index = LETTERS

pd.DataFrame({"letter": LETTERS, "count": counts.values}).to_csv(
    os.path.join(TABLE_DIR, "class_distribution.csv"), index=False)

plt.figure(figsize=(12, 4))
colors = ["#E07B39" if (c == counts.max() or c == counts.min()) else "#2E6FB7" for c in counts]
plt.bar(counts.index, counts.values, color=colors)
plt.title("Class distribution (number of images per letter)")
plt.xlabel("Letter"); plt.ylabel("Count")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "01_class_distribution.pdf"), dpi=150)
plt.show()

print("Most common:", counts.idxmax(), int(counts.max()))
print("Least common:", counts.idxmin(), int(counts.min()))
print("Imbalance ratio (max/min): %.1f" % (counts.max() / counts.min()))

In [ ]:
# ---- one sample image per class ----
fig, axes = plt.subplots(3, 9, figsize=(12, 4.5))
axes = axes.ravel()
for i in range(27):
    axes[i].axis("off")
    if i < 26:
        idx = np.where(labels == i)[0][0]        
        axes[i].imshow(X[idx].reshape(28, 28), cmap="gray_r")
        axes[i].set_title(LETTERS[i])
fig.suptitle("One sample image per class (28x28 grayscale)")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "02_sample_letters.pdf"), dpi=150, bbox_inches="tight")
plt.show()

## 3. Preprocessing: reshape and normalisation

Each flat 784-value row is reshaped into a 28x28x1 image. The custom CNN keeps the native
28x28 size, so no resizing is needed there; the transfer-learning model resizes to 32x32 later
(Section 10).

In [ ]:
X = X.reshape(-1, 28, 28, 1)
print("Image tensor shape:", X.shape, "| dtype:", X.dtype)

if SAMPLE_FRACTION < 1.0:
    idx, _ = train_test_split(np.arange(len(labels)), train_size=SAMPLE_FRACTION,
                              stratify=labels, random_state=SEED)
    X, labels = X[idx], labels[idx]
    print("Subsampled to:", X.shape)

## 4. Train / validation / test split (stratified 70 / 15 / 15)

In [ ]:
# First split off the test set (15%)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, labels, test_size=0.15, stratify=labels, random_state=SEED)

# Then split the remaining 85% into train (70% of total) and validation (15% of total)
# 0.15 / 0.85 = 0.1765 of the remainder goes to validation
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.1765, stratify=y_temp, random_state=SEED)

del X, labels, X_temp, y_temp; gc.collect()

print("Train:", X_train.shape, "| Val:", X_val.shape, "| Test:", X_test.shape)
# verify proportions are preserved (should be nearly identical across splits)
for name, y in [("train", y_train), ("val", y_val), ("test", y_test)]:
    frac = pd.Series(y).value_counts(normalize=True).sort_index().round(3).values
    print(name, "class fraction (first 5 letters):", frac[:5])

## 5. Data augmentation

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(0.03),                       # +/- ~11 degrees
    tf.keras.layers.RandomTranslation(0.10, 0.10),              # +/- 10% shift
    tf.keras.layers.RandomZoom(0.10),                           # +/- 10% zoom
], name="data_augmentation")

# Visualise augmentation on one training image
sample = tf.cast(X_train[0:1], tf.float32) / 255.0
plt.figure(figsize=(9, 2.2))
for i in range(6):
    aug = data_augmentation(sample, training=True)
    plt.subplot(1, 6, i + 1)
    plt.imshow(tf.squeeze(aug), cmap="gray_r"); plt.axis("off")
plt.suptitle("Same training image after random augmentation (label: %s)" % LETTERS[y_train[0]])
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "03_augmentation_examples.pdf"), dpi=150, bbox_inches="tight")
plt.show()

## 6. Input pipelines (tf.data)

In [ ]:
def make_ds(Xa, ya, training=False, batch=BATCH):
    ds = tf.data.Dataset.from_tensor_slices((Xa, ya))
    if training:
        ds = ds.shuffle(min(len(Xa), 10000), seed=SEED)
    ds = ds.map(lambda x, y: (tf.cast(x, tf.float32), y), num_parallel_calls=AUTOTUNE)
    return ds.batch(batch).prefetch(AUTOTUNE)

train_ds = make_ds(X_train, y_train, training=True)
val_ds   = make_ds(X_val,   y_val,   training=False)
test_ds  = make_ds(X_test,  y_test,  training=False)

def make_ds_tl(Xa, ya, training=False, batch=BATCH_TL):
    ds = tf.data.Dataset.from_tensor_slices((Xa, ya))
    if training:
        ds = ds.shuffle(min(len(Xa), 10000), seed=SEED)
    def prep(x, y):
        x = tf.cast(x, tf.float32)
        x = tf.image.resize(x, [IMG_TL, IMG_TL])              
        x = tf.image.grayscale_to_rgb(x)                       
        x = tf.keras.applications.mobilenet_v2.preprocess_input(x)  
        return x, y
    ds = ds.map(prep, num_parallel_calls=AUTOTUNE)
    return ds.batch(batch).prefetch(AUTOTUNE)

train_ds_tl = make_ds_tl(X_train, y_train, training=True)
val_ds_tl   = make_ds_tl(X_val,   y_val,   training=False)
test_ds_tl  = make_ds_tl(X_test,  y_test,  training=False)
print("Pipelines ready.")

## 7. Custom CNN model

In [ ]:
def build_cnn():
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(28, 28, 1)),
        tf.keras.layers.Rescaling(1.0 / 255),                  # normalise 0-255 -> 0-1
        data_augmentation,                                     # train-time only
        tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu", name="conv1"),
        tf.keras.layers.MaxPooling2D(2),
        tf.keras.layers.Conv2D(64, 3, padding="same", activation="relu", name="conv2"),
        tf.keras.layers.MaxPooling2D(2),
        tf.keras.layers.Conv2D(128, 3, padding="same", activation="relu", name="conv3"),
        tf.keras.layers.MaxPooling2D(2),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dropout(0.4),
        tf.keras.layers.Dense(256, activation="relu"),
        tf.keras.layers.Dense(NUM_CLASSES, activation="softmax"),
    ], name="custom_cnn")
    return model

cnn = build_cnn()
cnn.summary()

## 8. Compile and train the custom CNN

In [ ]:
cnn.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
            loss="sparse_categorical_crossentropy",
            metrics=["accuracy"])

class_weight = None
if USE_CLASS_WEIGHTS:
    classes = np.unique(y_train)
    weights = compute_class_weight("balanced", classes=classes, y=y_train)
    class_weight = dict(zip(classes.tolist(), weights.tolist()))
    print("Example class weights:", {LETTERS[k]: round(v, 2) for k, v in list(class_weight.items())[:5]})

cnn_ckpt = os.path.join(MODEL_DIR, "cnn_best.keras")
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5),
    tf.keras.callbacks.ModelCheckpoint(cnn_ckpt, monitor="val_loss", save_best_only=True),
]

history = cnn.fit(train_ds, validation_data=val_ds, epochs=EPOCHS,
                  class_weight=class_weight, callbacks=callbacks, verbose=1)

## 9. CNN training curves

In [ ]:
def plot_curves(hist, title, fname):
    h = hist.history
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
    a1.plot(h["accuracy"], label="train"); a1.plot(h["val_accuracy"], label="val")
    a1.set_title(title + " - accuracy"); a1.set_xlabel("epoch"); a1.set_ylabel("accuracy"); a1.legend()
    a2.plot(h["loss"], label="train"); a2.plot(h["val_loss"], label="val")
    a2.set_title(title + " - loss"); a2.set_xlabel("epoch"); a2.set_ylabel("loss"); a2.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, fname), dpi=150)
    plt.show()

plot_curves(history, "Custom CNN", "04_cnn_curves.pdf")

## 10. Transfer-learning model (MobileNetV2)

In [ ]:
def build_mobilenet():
    base = tf.keras.applications.MobileNetV2(
        input_shape=(IMG_TL, IMG_TL, 3), include_top=False, weights="imagenet")
    base.trainable = False                                     # stage 1: freeze the base
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(IMG_TL, IMG_TL, 3)),
        base,
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dropout(0.4),
        tf.keras.layers.Dense(256, activation="relu"),
        tf.keras.layers.Dense(NUM_CLASSES, activation="softmax"),
    ], name="mobilenetv2_transfer")
    return model, base

mnet, base = build_mobilenet()
mnet.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
             loss="sparse_categorical_crossentropy", metrics=["accuracy"])

mnet_ckpt = os.path.join(MODEL_DIR, "mobilenet_best.keras")
tl_callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(mnet_ckpt, monitor="val_loss", save_best_only=True),
]

# Stage 1: train the head
hist_tl = mnet.fit(train_ds_tl, validation_data=val_ds_tl,
                   epochs=HEAD_EPOCHS, class_weight=class_weight,
                   callbacks=tl_callbacks, verbose=1)

In [ ]:
# Stage 2 : fine-tune the top layers of the base at a low learning rate
if DO_FINETUNE:
    base.trainable = True
    for layer in base.layers[:-30]:        # keep the lower layers frozen
        layer.trainable = False
    mnet.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
                 loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    hist_ft = mnet.fit(train_ds_tl, validation_data=val_ds_tl,
                       epochs=FT_EPOCHS, class_weight=class_weight,
                       callbacks=tl_callbacks, verbose=1)

## 11. Evaluation on the held-out test set

In [ ]:
cnn_test_loss,  cnn_test_acc  = cnn.evaluate(test_ds,  verbose=0)
mnet_test_loss, mnet_test_acc = mnet.evaluate(test_ds_tl, verbose=0)
print("Custom CNN   - test accuracy: %.4f" % cnn_test_acc)
print("MobileNetV2  - test accuracy: %.4f" % mnet_test_acc)

cnn_prob  = cnn.predict(test_ds,  verbose=0)
mnet_prob = mnet.predict(test_ds_tl, verbose=0)
cnn_pred  = cnn_prob.argmax(axis=1)
mnet_pred = mnet_prob.argmax(axis=1)

## 12. Classification report and confusion matrix (custom CNN)

In [ ]:
print("Classification report - Custom CNN\n")
print(classification_report(y_test, cnn_pred, target_names=LETTERS, digits=3))

report_dict = classification_report(y_test, cnn_pred, target_names=LETTERS,
                                     output_dict=True, zero_division=0)
pd.DataFrame(report_dict).transpose().to_csv(
    os.path.join(TABLE_DIR, "cnn_classification_report.csv"))

In [ ]:
cm = confusion_matrix(y_test, cnn_pred, normalize="true")
plt.figure(figsize=(11, 9))
sns.heatmap(cm, xticklabels=LETTERS, yticklabels=LETTERS, cmap="Blues",
            vmin=0, vmax=1, square=True, cbar_kws={"shrink": 0.8})
plt.title("Custom CNN - normalised confusion matrix")
plt.xlabel("Predicted"); plt.ylabel("True")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "05_confusion_matrix.pdf"), dpi=150)
plt.show()

cm_counts = confusion_matrix(y_test, cnn_pred)
pairs = []
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        if i != j and cm_counts[i, j] > 0:
            pairs.append((LETTERS[i], LETTERS[j], cm_counts[i, j]))
pairs.sort(key=lambda t: t[2], reverse=True)
pd.DataFrame(pairs, columns=["true", "predicted", "count"]).to_csv(
    os.path.join(TABLE_DIR, "confused_pairs.csv"), index=False)
print("Most confused (true -> predicted, count):")
for t in pairs[:10]:
    print("  %s -> %s : %d" % t)

## 13. Per-class recall (imbalance analysis, RQ3)

In [ ]:
rec = recall_score(y_test, cnn_pred, average=None)
macro = rec.mean()
pd.DataFrame({"letter": LETTERS, "recall": rec}).to_csv(
    os.path.join(TABLE_DIR, "per_class_recall.csv"), index=False)
order_counts = pd.Series(y_test).value_counts()
thresh = order_counts.quantile(0.25)
bar_colors = ["#E07B39" if order_counts.get(i, 0) <= thresh else "#2E6FB7" for i in range(NUM_CLASSES)]

plt.figure(figsize=(12, 4))
plt.bar(LETTERS, rec, color=bar_colors)
plt.axhline(macro, color="#444", ls="--", label="macro avg = %.3f" % macro)
plt.ylim(min(0.7, rec.min() - 0.05), 1.0)
plt.title("Per-class recall (orange = rare letters)")
plt.xlabel("Letter"); plt.ylabel("Recall"); plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "06_per_class_recall.pdf"), dpi=150)
plt.show()

worst = np.argsort(rec)[:5]
print("Lowest-recall letters:", [(LETTERS[i], round(rec[i], 3)) for i in worst])

## 14. ROC curves and AUC (one-vs-rest)

In [ ]:
y_test_bin = label_binarize(y_test, classes=list(range(NUM_CLASSES)))

# micro-average
fpr_micro, tpr_micro, _ = roc_curve(y_test_bin.ravel(), cnn_prob.ravel())
auc_micro = auc(fpr_micro, tpr_micro)

# macro-average
fprs, tprs, aucs = [], [], []
for i in range(NUM_CLASSES):
    f, t, _ = roc_curve(y_test_bin[:, i], cnn_prob[:, i])
    fprs.append(f); tprs.append(t); aucs.append(auc(f, t))
all_fpr = np.unique(np.concatenate(fprs))
mean_tpr = np.zeros_like(all_fpr)
for i in range(NUM_CLASSES):
    mean_tpr += np.interp(all_fpr, fprs[i], tprs[i])
mean_tpr /= NUM_CLASSES
auc_macro = auc(all_fpr, mean_tpr)

plt.figure(figsize=(6, 5))
plt.plot(fpr_micro, tpr_micro, label="micro-average (AUC = %.3f)" % auc_micro)
plt.plot(all_fpr, mean_tpr, label="macro-average (AUC = %.3f)" % auc_macro)
plt.plot([0, 1], [0, 1], "k--", lw=1)
plt.title("Custom CNN - ROC (one-vs-rest)")
plt.xlabel("False positive rate"); plt.ylabel("True positive rate"); plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "07_roc_curves.pdf"), dpi=150)
plt.show()
pd.DataFrame({"letter": LETTERS, "auc": aucs}).to_csv(
    os.path.join(TABLE_DIR, "per_class_auc.csv"), index=False)
print("Mean per-class AUC: %.4f" % np.mean(aucs))

## 15. Model comparison (custom CNN vs MobileNetV2, RQ4)

In [ ]:
def f1_scores(y_true, y_pred):
    r = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    return r["accuracy"], r["macro avg"]["f1-score"], r["weighted avg"]["f1-score"]

cnn_acc, cnn_mf1, cnn_wf1   = f1_scores(y_test, cnn_pred)
mnet_acc, mnet_mf1, mnet_wf1 = f1_scores(y_test, mnet_pred)

comparison = pd.DataFrame({
    "Model":        ["Custom CNN", "MobileNetV2 (transfer)"],
    "Accuracy":     [cnn_acc,  mnet_acc],
    "Macro F1":     [cnn_mf1,  mnet_mf1],
    "Weighted F1":  [cnn_wf1,  mnet_wf1],
}).round(4)
comparison.to_csv(os.path.join(TABLE_DIR, "model_comparison.csv"), index=False)
print(comparison.to_string(index=False))

# grouped bar chart
metrics = ["Accuracy", "Macro F1", "Weighted F1"]
x = np.arange(len(metrics)); w = 0.35
plt.figure(figsize=(7.5, 4))
plt.bar(x - w/2, comparison.iloc[0, 1:].values, w, label="Custom CNN", color="#2E6FB7")
plt.bar(x + w/2, comparison.iloc[1, 1:].values, w, label="MobileNetV2", color="#E07B39")
plt.xticks(x, metrics); plt.ylim(0, 1.0); plt.ylabel("Score"); plt.legend()
plt.title("Model comparison")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "08_model_comparison.pdf"), dpi=150)
plt.show()

## 16. Grad-CAM (what the model looks at, RQ5)

In [ ]:

def build_grad_model(model, last_conv_layer_name="conv3"):
    inp = tf.keras.Input(shape=(28, 28, 1))
    x = inp
    conv_output = None
    for layer in model.layers:           
        x = layer(x)
        if layer.name == last_conv_layer_name:
            conv_output = x              
    return tf.keras.models.Model(inp, [conv_output, x])

grad_model = build_grad_model(cnn, "conv3")

def make_gradcam_heatmap(img_array, grad_model, pred_index=None):
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(img_array, training=False)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]
    grads = tape.gradient(class_channel, conv_out)
    pooled = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_out = conv_out[0]
    heatmap = tf.squeeze(conv_out @ pooled[..., tf.newaxis])
    heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

# pick a few examples to display
show_idx = [np.where(y_test == c)[0][0] for c in [0, 4, 12, 16, 17]]   # A, E, M, Q, R
plt.figure(figsize=(11, 2.6))
for k, idx in enumerate(show_idx):
    img = X_test[idx:idx+1].astype("float32")         
    heat = make_gradcam_heatmap(img, grad_model)
    heat = tf.image.resize(heat[..., None], [28, 28]).numpy().squeeze()
    plt.subplot(1, len(show_idx), k + 1)
    plt.imshow(X_test[idx].reshape(28, 28), cmap="gray_r")
    plt.imshow(heat, cmap="jet", alpha=0.5)
    plt.title("%s -> %s" % (LETTERS[y_test[idx]], LETTERS[cnn_pred[idx]]))
    plt.axis("off")
plt.suptitle("Grad-CAM overlays (custom CNN)")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "09_gradcam.pdf"), dpi=150, bbox_inches="tight")
plt.show()

## 17. Error analysis

A look at some misclassified test images. These usually fall on letter pairs that look alike
when handwritten.

In [ ]:
wrong = np.where(cnn_pred != y_test)[0]
print("Total misclassified: %d / %d (%.2f%%)" % (len(wrong), len(y_test), 100*len(wrong)/len(y_test)))

sel = wrong[:12]
plt.figure(figsize=(11, 3.2))
for k, idx in enumerate(sel):
    plt.subplot(2, 6, k + 1)
    plt.imshow(X_test[idx].reshape(28, 28), cmap="gray_r")
    plt.title("T:%s P:%s" % (LETTERS[y_test[idx]], LETTERS[cnn_pred[idx]]), fontsize=9)
    plt.axis("off")
plt.suptitle("Examples of misclassified letters (T = true, P = predicted)")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "10_error_analysis.pdf"), dpi=150, bbox_inches="tight")
plt.show()

## 18. Save models, figures, and metrics

Everything is written to the Kaggle working directory so it can be downloaded: figures as **PDF** in `figures/`, tables as **CSV** in `tables/`, trained
models in `models/`, and an overall `metrics_summary.json`.

In [ ]:
# Save the final models
cnn.save(os.path.join(MODEL_DIR, "cnn_model.keras"))
mnet.save(os.path.join(MODEL_DIR, "mobilenetv2_model.keras"))

# Save a compact metrics summary
metrics_summary = {
    "custom_cnn":  {"test_accuracy": float(cnn_acc),  "macro_f1": float(cnn_mf1),  "weighted_f1": float(cnn_wf1),  "auc_macro": float(auc_macro)},
    "mobilenetv2": {"test_accuracy": float(mnet_acc), "macro_f1": float(mnet_mf1), "weighted_f1": float(mnet_wf1)},
    "config": {"seed": SEED, "batch": BATCH, "epochs": EPOCHS,
               "img_tl": IMG_TL, "class_weights": USE_CLASS_WEIGHTS, "sample_fraction": SAMPLE_FRACTION},
}
with open(os.path.join(WORK, "metrics_summary.json"), "w") as f:
    json.dump(metrics_summary, f, indent=2)
print(json.dumps(metrics_summary, indent=2))

print("\nSaved files in", WORK, ":")
for p in sorted(glob.glob(os.path.join(WORK, "**", "*"), recursive=True)):
    if os.path.isfile(p):
        print(" ", p)